In [ ]:
# 数据分析模块主要函数声明
# query_data(start_time=None, end_time=None, username=None, board_name=None, db_path="kf.db", reverse=False)
# ↑ 按填入条件查询帖子，返回值有两个：①散装replies；②结构topics。
# ↑ ①所有符合条件的回复贴按回复时间顺序排序的列表，相较原始数据新增了reply_length字段（utf8字节数）
# ↑ ②主题帖新增reply_list/username/complete字段，reply_list中只含有该主题贴中所有符合条件的回复贴
# ↑ ②username用来指示该主题贴的发帖人，complete用来指示你是否解锁了该主题贴中所有的购买框和权限框

In [ ]:
import json, csv, os, re, sqlite3, time, collections, bisect
from datetime import date, datetime, timedelta
from collections import Counter
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import numpy as np
from kf_analysis import analytics
from kf_analysis.coordinator import KFanalysis
from kf_analysis.utils import load_config
# 设定图表中文字体，默认为更纱黑体Gothic
plt.rcParams["font.sans-serif"] = ["Sarasa Gothic SC"]
plt.rcParams["axes.unicode_minus"] = False
start_time, end_time = "2026-08-01 00:00:00", "2026-08-31 23:59:59"
replies, topics = analytics.query_data(start_time=start_time, end_time=end_time)

In [ ]:
# 简述A：统计期间，总活跃主题数，总新增回复数，总参与人数
# 简述B：天平均活跃主题数，天平均新增回复数，天平均参与人数
# 统计期间新增回复量热力图，天发言用户量折线图
# 统计期间各板块活跃主题量柱状图，统计期间各板块新增回复量柱状图
posters = len({r["username"] for r in replies})
days = (analytics.to_datetime(end_time).date() - analytics.to_datetime(start_time).date()).days + 1
print(f"统计期间：[{start_time[:-9]}, {end_time[:-9]}, {days}]\n"
      f"{posters} 账号参与了 {len(topics)} 主题的活跃，产生 {len(replies)} 回复。")
day_counts = Counter()
daily_topics = {}
daily_users = {}
for r in replies:
    d = analytics.to_datetime(r["reply_time"]).date()
    day_counts[d] += 1
    daily_topics.setdefault(d, set()).add(r["topic_id"])
    if r["username"]:
        daily_users.setdefault(d, set()).add(r["username"])
avg_topics = sum(len(s) for s in daily_topics.values()) / days
avg_users = sum(len(s) for s in daily_users.values()) / days
print(f"平均每天 {avg_topics:.1f} 主题活跃，{len(replies)/days:.1f} 回复新增，{avg_users:.1f} 账号发言。")
print("#1 本项目将主题第零楼也视作回复之一\n"
      "#2 主题统计口径不再是“统计期间新增的”，而是“统计期间活跃过的”\n"
      "#3 天均主题活跃量、天均发言账号量：(ΣTd)/D、(ΣUd)/D")
# calendar_heatmap函数可以通过cell_height参数调节格子高度
analytics.calendar_heatmap(day_counts, start_time, end_time, title="每天新增回复数量热力图", save="每天新增回复数量热力图")
boardlist = json.load(open("kf_analysis/configure.json", "r", encoding="utf-8"))["boardlist"]
board_order = {fid: i for i, (_, fid) in enumerate(boardlist)}
board_names = {t["board_id"]: t["board_name"] for t in topics}
topic_board_counts = Counter(t["board_id"] for t in topics)
reply_board_counts = Counter(r["board_id"] for r in replies)
boards = sorted(topic_board_counts, key=lambda fid: board_order.get(fid, len(board_order)))
labels = [board_names[fid] for fid in boards]
daily_dates = sorted(daily_users)
date_labels = [d.strftime("%m%d") for d in daily_dates]
analytics.output_plot_line(date_labels, [len(daily_users[d]) for d in daily_dates], title="每天发言用户数量", save="每天发言用户数量", color="skyblue", figsize=(16,3.5))
analytics.output_plot_bar(labels, [topic_board_counts[fid] for fid in boards], title="各板块活跃主题数量", save="各板块活跃主题数量", color="lightpink", figsize=(16,4.5))
analytics.output_plot_bar(labels, [reply_board_counts[fid] for fid in boards], title="各板块新增回复数量", save="各板块新增回复数量", color="mediumpurple", figsize=(16,4.5))

In [ ]:
# 用户活跃属性排行：输出为表格
# 用户活跃属性排行：回复数量 / 回复字节数 / 活跃天数比例
# exclude_boards用于在回复字节数统计时排除部分板块
exclude_boards = ["Galgame 网络硬盘区", "ACG音乐资源共享区", "CG画册资源共享区", "无限制资源区",
                  "动画资源共享区", "漫画轻小说共享区", "LIVE类资源分享区", "Galgame BitTorrent区", "GAL本子区"]
top_n = 100
reply_count = Counter()
reply_bytes = Counter()
active_days = {}
for r in replies:
    if not r["username"]: continue
    reply_count[r["username"]] += 1
    if r["board_name"] not in exclude_boards:
        reply_bytes[r["username"]] += r["reply_length"]
    active_days.setdefault(r["username"], set()).add(analytics.to_datetime(r["reply_time"]).date())
top_post = reply_count.most_common(top_n)
top_bytes = reply_bytes.most_common(top_n)
active_ratio = sorted(((n, len(d), len(d) / days * 100) for n, d in active_days.items()), key=lambda x: x[2], reverse=True)[:top_n]
def show(title, rows):
    print(f"\n{title} TOP{top_n}\n" + "="*30)
    for rank, (name, val) in enumerate(rows, 1):
        print(f"第{rank:>3}名 | {val} | {name}")
print("#1 回复字节数为回复内容的UTF8编码字节数。\n"
      "#2 回复字节数统计对象不包括九个资源区。")
show("回复数量", [(n, f"{c:>4} 回复") for n, c in top_post])
show("回复字节数", [(n, f"{c:>7} 字节") for n, c in top_bytes])
show("活跃天数比例", [(n, f"{d:>2}/{days} 天（{r:.2f}%）") for n, d, r in active_ratio])
with open("user_ranking.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["rank", "usernameA", "数量", "usernameB", "字节量", "usernameC", "天数", "比例%"])
    for i in range(top_n):
        a_name, a_val = top_post[i] if i < len(top_post) else ("", "")
        b_name, b_val = top_bytes[i] if i < len(top_bytes) else ("", "")
        c_name, c_days, c_ratio = active_ratio[i] if i < len(active_ratio) else ("", "", "")
        writer.writerow([i + 1, a_name, a_val, b_name, b_val, c_name, c_days,
                         round(c_ratio, 2) if c_ratio else ""])

In [ ]:
# 用户活跃属性排行：输出为图像
rows = []
for i in range(top_n):
    a = top_post[i] if i < len(top_post) else ("", "")
    b = top_bytes[i] if i < len(top_bytes) else ("", "")
    c = active_ratio[i] if i < len(active_ratio) else ("", "", "")
    rows.append([i + 1, a[0], f"{a[1]:g}", b[0], f"{b[1]:g}", c[0], f"{c[1]:g}", f"{round(c[2], 2):g}"])
COLW = [80, 141, 79, 141, 79, 141, 79, 79]
ALIGN = ["l", "l", "r", "l", "r", "l", "r", "r"]
BG = ["#FFFFFF", "#C6E0B4", "#C6E0B4", "#BDD7EE", "#BDD7EE", "#F8CBAD", "#F8CBAD", "#F8CBAD"]
HEAD = ["rank", "usernameA", "数量", "usernameB", "字节量", "usernameC", "天数", "比例%"]
font = ImageFont.truetype("C:/Windows/Fonts/msyh.ttc", 16)
im = Image.new("RGB", (sum(COLW) + len(COLW), 23 * (len(rows) + 1) + 1), "#FFFFFF")
dr = ImageDraw.Draw(im)
for k, cells in enumerate([HEAD] + rows):
    y = 23 * k + 1
    for i, val in enumerate(cells):
        x0 = sum(COLW[:i]) + i
        dr.rectangle([x0, y, x0 + COLW[i] - 1, y + 21], fill=BG[i])
        anchor = "lm" if ALIGN[i] == "l" else "rm"
        tx = x0 + 5 if ALIGN[i] == "l" else x0 + COLW[i] - 5
        dr.text((tx, y + 11), str(val), font=font, fill="#000000", anchor=anchor)
for i in range(len(COLW) + 1):
    x = sum(COLW[:i]) + i - 1 if i else 0
    dr.line([x, 0, x, im.height - 1], fill="#000000")
for j in range(len(rows) + 2):
    dr.line([0, 23 * j, im.width - 1, 23 * j], fill="#000000")
im.save("user_ranking.png")


In [ ]:
# 账号新增与留存分析：前置单元A
# 将未入库的账号补入hp.db以提升分析精度
kf = KFanalysis(load_config())
hpc = sqlite3.connect("hp.db")
done = {r[0] for r in hpc.execute("SELECT uid FROM homepage")}
todo = sqlite3.connect("kf.db").execute("SELECT DISTINCT homepage_id, homepage_sf FROM reply WHERE homepage_id IS NOT NULL").fetchall()
new = [(u, s) for u, s in todo if u not in done]
print(f"总数 {len(todo)}，已录 {len(done)}，预计补录 {len(new)}。")
for uid, sf in new:
    data = False
    for i in range(3):
        try: data = kf.get_homepage(uid, sf, db=True)
        except: print(f"{uid}请求异常第{i}次。")
        if data is not False: break
        time.sleep(1.5)
    if data is False: print(f"{uid}补录失败。")
    time.sleep(1.5)

In [ ]:
# 账号新增与留存分析：前置单元B
# 根据kf.db现存全量数据统计“时段回复数量分布（三小时移动平均）”
# 不仅是单独统计项目，也将作为注册时间估算的权重来源
hours = {}
for t, in sqlite3.connect("kf.db").execute("SELECT reply_time FROM reply"):
    h = datetime.fromtimestamp(t).hour
    hours[h] = hours.get(h, 0) + 1
w_hour = [hours.get(h, 0) for h in range(24)]
w_smooth = [(w_hour[(h - 1) % 24] + w_hour[h] + w_hour[(h + 1) % 24]) / 3 for h in range(24)]
analytics.output_plot_bar([f"{h}时" for h in range(24)], w_smooth, title="时段回复数量分布", save="回复时段数量分布", color="lightblue", rotation=0)

In [ ]:
# 账号新增与留存分析：估算时刻T用户数量
# 假设我们有一张24小时热度分布权重表，表中元素相加为1
# 当已知点数量为1，代表将一天分成了2份，接下来我们需要在权重表中找到从左向右加和到恰好等于50%的点
# 当已知点数量为n，代表将一天分成了n+1份，接下来我们需要在权重表中分别找到从左向右加和恰好等于k/(n+1)处的点
# （实际上是先定位到小时然后在小时内线性插值，关于此处精度的改善可以从权重表入手，在上一个cell提高权重表时间粒度，但这也意味着更长的计算时间）
# 为避免重复计算，先找出单日已知点数量的最大值N，然后前置地分别算出当已知点数量为1到N时，每个点的对应时刻，后面只需要查表赋值就好
# 为了得到T时刻的uid最大值，我们需要找到已知点中早于T与晚于T的最近点，然后根据Ut=Ua+(Ub-Ua)×Wa得到结果，Wa指时刻A到时刻T占时刻A到时刻B的权重比例
pts = [(u, date(*map(int, d.split("-")))) for u, d in sqlite3.connect("hp.db").execute("SELECT uid, regdate FROM homepage ORDER BY uid")]
by_day = collections.defaultdict(list)
for u, d in pts: by_day[d].append(u)
day_total = sum(w_smooth)
STEP = 24 / len(w_smooth)
cumw = [0.0]
for i in range(len(w_smooth)): cumw.append(cumw[-1] + w_smooth[i])
def inv(q):
    t = q * day_total
    k = bisect.bisect_left(cumw, t) - 1
    return (k + (t - cumw[k]) / w_smooth[k]) * STEP
def at0(d): return datetime.combine(d, datetime.min.time())
max_m = max(len(us) for us in by_day.values())
hours_tab = {m: [inv((i + 1) / (m + 1)) for i in range(m)] for m in range(1, max_m + 1)}
pint = []
for d in sorted(by_day):
    us = by_day[d]
    ht = hours_tab[len(us)]
    pint += [(u, at0(d) + timedelta(hours=ht[i])) for i, u in enumerate(us)]
ts = [t for _, t in pint]
T0 = at0(pts[0][1])
def W(T):
    dt = T - T0
    n, f = dt.days, dt.seconds / 3600 + dt.microseconds / 3.6e9
    k = int(f / STEP)
    return n * day_total + cumw[k] + (f - k * STEP) / STEP * w_smooth[k]
def est_max_uid(T):
    i = bisect.bisect_left(ts, T) - 1
    if i < 0: return pts[0][0] - 1
    if i >= len(pint) - 1: return pts[-1][0]
    (u_a, t_a), (u_b, t_b) = pint[i], pint[i + 1]
    return u_a + (u_b - u_a) * (W(T) - W(t_a)) / (W(t_b) - W(t_a))
nd = (pts[-1][1] - pts[0][1]).days + 1
ubar = [est_max_uid(T0 + timedelta(days=k)) for k in range(nd + 1)]
day_inc = [(pts[0][1] + timedelta(days=k), ubar[k + 1] - ubar[k]) for k in range(nd)]
print("总量守恒:", ubar[-1] - ubar[0], "==", pts[-1][0] - pts[0][0] + 1)
# 天粒度新增账号估算（纵轴截断1000并标注越界值）
analytics.plot_daily_bars(day_inc, save="每日新增账号估算", ylim=1000)
# 月度新增账号折线图（仅输出锚点充足的月份）
# 从最新月份向左寻找第一个已知点数量大于10的月份A，从月份A向左框定12个月
# 10已知点的限制在月粒度的估算上误差能够接收，但在天粒度的估算上误差会很大
month_delta = collections.Counter()
for d, v in day_inc: month_delta[(d.year, d.month)] += v
month_delta = sorted(month_delta.items())
A = max(ym for ym, n in sqlite3.connect("hp.db").execute("SELECT substr(regdate, 1, 7), COUNT(*) FROM homepage WHERE ok=1 GROUP BY 1") if n > 10)
i = next(k for k, ((y, m), _) in enumerate(month_delta) if f"{y}-{m:02d}" == A)
win = month_delta[i - 11:i + 1]
analytics.output_plot_line([f"{y}-{m:02d}" for (y, m), _ in win], [v for _, v in win], title=f"月度新增账号折线图", save="月度新增账号折线图", color="tomato", figsize=(16, 4.5))

In [ ]:
# 统计期间活跃账号的留存情况
hp_year = {u: y[:4] for y, u in hpc.execute("SELECT regdate, uid FROM homepage WHERE ok=1")}
active = {r["homepage_id"] for r in replies if r.get("homepage_id")}
years = Counter(hp_year[u] for u in active if u in hp_year)
years = {str(y): years.get(str(y), 0) for y in range(int(min(hp_year.values())), int(max(hp_year.values())) + 1)}
analytics.output_plot_bar([str(y) for y, _ in years.items()], [c for _, c in years.items()], title="本月活跃用户注册年份分布", save="本月活跃用户注册年份分布", color="lightsalmon", rotation=0)

In [ ]:
top_n = 50
hb_color = "#FF0000"
normal_color = "#000000"
hb_title_pat = re.compile(r"hb", re.I)
hb_reply_pat = re.compile(r"(感谢|谢谢|多谢|有).{0,10}hb", re.I)
topic_day_counts = Counter()
for tids in daily_topics.values(): topic_day_counts.update(tids)
hot_stats = [(len(t["reply_list"]), topic_day_counts[t["topic_id"]], t) for t in topics]
def is_hb(t):
    return bool(hb_title_pat.search(t["topic_title"]) or any(hb_reply_pat.search(r["reply_text"] or "") for r in t["reply_list"]))
def show_hot(title, rows):
    print(f"\n{title} TOP{top_n}")
    for rank, (val, t) in enumerate(rows, 1):
        line = f"[b]no. {rank}: {val}: [/b][url={t['topic_url']}]{t['topic_title']}[/url]"
        print(f"[color={hb_color if is_hb(t) else normal_color}]{line}[/color]")
print("#1 通过标题与回复列表自动判断是否为hb相关贴，若是则以红色标注。")
show_hot("回复数量热帖榜", [(f"{c} replies", t) for c, days, t in sorted(hot_stats, key=lambda x: x[0], reverse=True)[:top_n]])
show_hot("讨论持续天数热帖榜", [(f"{days} days", t) for c, days, t in sorted(hot_stats, key=lambda x: x[1], reverse=True)[:top_n]])

In [ ]:
# 统计期间内，用户在九个资源区中的新增主题数量（总体与自购）
new_res_topics = [t for t in topics
                  if t["board_name"] in exclude_boards and t["username"]
                  and t["reply_list"] and t["reply_list"][0]["floor"] == 0]
post_counts = Counter(t["username"] for t in new_res_topics)
self_buy_counts = Counter(t["username"] for t in new_res_topics if "自购" in t["topic_title"])
def fmt_counts(counter):
    items = [f"{name} ({cnt})" for name, cnt in counter.most_common()]
    return "\n".join(", ".join(items[i:i + 5]) for i in range(0, len(items), 5))
print("总体数量：\n" + fmt_counts(post_counts) + "\n")
print("自购数量：\n" + fmt_counts(self_buy_counts) + "\n")
# 导出统计期间内，两个求助区所有新增主题的tid
help_tids = [f"https://bbs.kfpromax.com/read.php?tid={t["topic_id"]}&sf={t["topic_sf"]}" for t in topics
             if t["board_name"] in ("寻求资源", "图片/作品出处询问版")
             and t["reply_list"] and t["reply_list"][0]["floor"] == 0]
with open("求助区新增主题列表.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(map(str, help_tids)))